<a href="https://colab.research.google.com/github/balla-a12/quant-equity-research/blob/main/06_composite_backtest_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 — Composite Scorer and Backtest

The payoff notebook. The four signals each produce a cross-sectional $0$–$100$ score;
this blends them into one conviction ranking and then asks the question the whole project
is built around — does combining diversified signals capture more edge than any single
one?

### How the blend works

The composite is a weighted sum of the component scores,

$$ \text{composite}_i \;=\; \sum_{s} w_s \cdot \text{score}_{s,i}, \qquad \sum_s w_s = 1 $$

with project weights congress $0.40$, contracts $0.30$, lobbying $0.20$, off-exchange
$0.10$. A ticker absent from a signal contributes $0$ to that term, which is deliberate:
it rewards corroboration. A name surfacing in congress buys *and* contracts accumulates
more weighted points than one appearing in a single signal. Since the weights sum to $1$
and each component is on $0$–$100$, the composite is also on $0$–$100$, read as conviction
points out of a possible $100$. The `n_signals` column shows how many components each name
actually appears in.

### What the backtest includes

Only the event signals carry usable history, so the **backtest blends congress, contracts,
and lobbying**. Off-exchange is a single daily snapshot with no history, so it stays out of
the historical test and joins only the present-day live ranking.

The backtest runs on a $2022$-start, weekly-rebalance schedule, matching the single-signal
backtest in `02` so the comparison is apples-to-apples. It is read through two lenses:

- a **statistical view** — the information coefficient, the single-versus-composite
  comparison, and the bucket profile, measured across all weekly rebalance dates;
- a **tradeable view** — a market-neutral long-short equity curve on non-overlapping
  monthly holds, with Sharpe, CAGR, and drawdown.

The event study also runs at two horizons, $21$ and $63$ trading days, as a robustness
check on whether the edge survives a longer hold.

A result worth stating up front: blending does not automatically beat the best single
signal. It helps when components are *diversifying*. When one signal dominates, mixing in
weaker ones can dilute it. The backtest is how that question gets answered rather than
assumed.


## Setup

This notebook builds on `01`–`05` being committed (it uses all four signals and the `02`
backtest engine from the repo). Defaults to mock; set `USE_LIVE = True` for live data.


In [1]:
!pip install -q quiverquant pandas numpy sqlalchemy plotly pyyaml requests yfinance scipy

import subprocess, os, sys, logging
from google.colab import userdata

GITHUB_USER = "balla-a12"
REPO = "quant-equity-research"

gh_token = userdata.get("GITHUB_TOKEN")
url = f"https://{gh_token}@github.com/{GITHUB_USER}/{REPO}.git"
r = subprocess.run(["git", "clone", url], capture_output=True, text=True)
if r.returncode == 0 or "already exists" in r.stderr:
    os.chdir(REPO)
subprocess.run(["git", "pull", "--rebase", "origin", "main"], capture_output=True, text=True)
logging.getLogger("yfinance").setLevel(logging.CRITICAL)
print("Working in:", os.getcwd())

try:
    QUIVER_TOKEN = userdata.get("QUIVER_API_KEY")
except Exception:
    QUIVER_TOKEN = None
USE_LIVE = True   # flip to True for live Quiver data + real prices

Working in: /content/quant-equity-research


## Write the composite module

The new module is `signals/composite.py`; the four signals and the backtest engine are
already in the cloned repo from the earlier notebooks.


In [2]:
open("src/quant_research/signals/composite.py", "w").write('"""Composite signal: blend the individual signals into one conviction ranking.\n\nEach component signal produces a cross-sectional 0-100 score. The composite takes a\nweighted sum of those scores, where a ticker absent from a signal contributes 0 to that\nterm. That choice is deliberate: it rewards corroboration. A name surfacing in congress\nbuys *and* government contracts accumulates more weighted points than one appearing in a\nsingle signal, which is exactly the conviction-stacking the project is testing.\n\nWith weights that sum to 1 and each component on 0-100, the composite score is itself on\n0-100, read as conviction points out of a possible 100 (a name topping every component\nat once). In practice no name tops all four, so the realized top scores sit well below\n100, and ``n_signals`` shows how many components a name actually appears in.\n\nFor the historical backtest, only the event signals carry usable history, so the\ncomposite is built from those; off-exchange is a present-day confirming layer that joins\nthe live ranking but stays out of the backtest.\n"""\nfrom datetime import date\nimport pandas as pd\n\nDEFAULT_WEIGHTS = {\n    "congress": 0.40,\n    "gov_contracts": 0.30,\n    "lobbying": 0.20,\n    "off_exchange": 0.10,\n}\n\n# maps a component name to the keyword its compute() uses for prefetched data\n_DATA_KWARG = {\n    "congress": "trades",\n    "gov_contracts": "contracts",\n    "lobbying": "filings",\n    "off_exchange": "flow",\n}\n\n\nclass CompositeSignal:\n    name = "composite"\n    description = "Weighted blend of the component signals into one conviction score."\n\n    def __init__(self, signals, weights=None):\n        """signals: dict of {component_name: signal_instance}.\n\n        Weights default to the project weights restricted to the components supplied, so\n        a three-signal backtest composite and a four-signal live composite share one class.\n        """\n        self.signals = signals\n        if weights is None:\n            weights = {k: v for k, v in DEFAULT_WEIGHTS.items() if k in signals}\n        self.weights = weights\n\n    def compute(self, as_of=None, prefetch=None):\n        as_of = pd.Timestamp(as_of or date.today())\n        cols = {}\n        for nm, sig in self.signals.items():\n            kw = {}\n            if prefetch and nm in prefetch and _DATA_KWARG.get(nm):\n                kw[_DATA_KWARG[nm]] = prefetch[nm]\n            res = sig.compute(as_of, **kw)\n            cols[nm] = res["score"] if len(res) else pd.Series(dtype=float)\n\n        wide = pd.DataFrame(cols)\n        for k in self.weights:\n            if k not in wide.columns:\n                wide[k] = pd.NA\n\n        w = pd.Series(self.weights)\n        present = wide[w.index].notna()\n        filled = wide[w.index].fillna(0.0)\n        score = filled.mul(w, axis=1).sum(axis=1)\n\n        out = wide.copy()\n        out["n_signals"] = present.sum(axis=1).astype(int)\n        out["score"] = score.round(1)\n        return out.sort_values("score", ascending=False)\n\n    def score_fn(self, prefetch=None):\n        """Return a ``score_fn(as_of) -> Series`` for the backtest engine."""\n        def _fn(as_of):\n            res = self.compute(as_of, prefetch=prefetch)\n            return res["score"] if len(res) else None\n        return _fn\n')
print("wrote composite.py")

wrote composite.py


In [3]:
!pip install -q -e .

import os, sys, importlib
src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
import quant_research; importlib.reload(quant_research)
print("Package ready:", quant_research.__version__)

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for quant-equity-research (pyproject.toml) ... done
Package ready: 0.1.0


## The live composite ranking

Build all four signals, blend them, and read today's top conviction names. The per-signal
columns show *where* each name's conviction comes from, and `n_signals` shows how broadly
it is corroborated.


In [4]:
import pandas as pd
from quant_research.ingestion.client import QuiverClient
from quant_research.signals.congress import CongressSignal
from quant_research.signals.govcontracts import GovContractsSignal
from quant_research.signals.lobbying import LobbyingSignal
from quant_research.signals.offexchange import OffExchangeSignal
from quant_research.signals.composite import CompositeSignal

client = QuiverClient(token=QUIVER_TOKEN, mock_history_days=600) if USE_LIVE \
         else QuiverClient(mock=True, mock_history_days=600)

signals_all = {
    "congress":      CongressSignal(client),
    "gov_contracts": GovContractsSignal(client),
    "lobbying":      LobbyingSignal(client),
    "off_exchange":  OffExchangeSignal(client),
}
composite = CompositeSignal(signals_all)

live = composite.compute()
print(f"{len(live)} tickers ranked | data: {'live' if USE_LIVE else 'mock'} | weights: {composite.weights}")
cols = ["congress", "gov_contracts", "lobbying", "off_exchange", "n_signals", "score"]
live[cols].head(12).round(1)

https://api.quiverquant.com/beta/live/congresstrading
1806 tickers ranked | data: live | weights: {'congress': 0.4, 'gov_contracts': 0.3, 'lobbying': 0.2, 'off_exchange': 0.1}


,congress,gov_contracts,lobbying,off_exchange,n_signals,score
ticker,,,,,,
MSFT,100.0,10.3,53.4,35.9,4,57.4
T,91.4,7.2,45.7,45.1,4,52.4
AAPL,72.3,NaN,54.7,48.4,3,44.7
AMD,78.3,NaN,19.7,51.1,3,40.4
DELL,NaN,100.0,22.7,52.1,3,39.8
F,NaN,97.3,23.9,45.1,3,38.5
GD,NaN,89.3,54.9,NaN,2,37.8
INTC,68.9,NaN,12.7,56.9,3,35.8
PLTR,5.8,63.9,40.1,44.5,4,34.0


### Visualize the conviction stack

Each bar's segments show how a name's composite score is built from its weighted
components, so a tall bar made of several colors is a broadly corroborated name.


In [5]:
import plotly.graph_objects as go

top = live.head(15).iloc[::-1]
parts = {"congress": 0.40, "gov_contracts": 0.30, "lobbying": 0.20, "off_exchange": 0.10}
fig = go.Figure()
for name, w in parts.items():
    fig.add_bar(y=top.index, x=top[name].fillna(0) * w, name=name, orientation="h")
fig.update_layout(barmode="stack", height=520, title="Composite conviction stack — top 15",
                  xaxis_title="weighted contribution to score")
fig.show()

## Backtest the composite

The historical test runs on the congress + contracts + lobbying blend (off-exchange has no
history). The schedule matches `02`: a $2022$ start with **weekly** rebalances, so the
event study has the same density of observations. Data is prefetched once and each signal
windows it per rebalance date. Prices come from yfinance on the live run; the mock run
injects a synthetic edge tied to the signals so the machinery is visible end to end.


In [6]:
from quant_research.backtest.engine import (event_study, long_short_curve,
                                            benchmark_curve, metrics)
import numpy as np

START = pd.Timestamp("2022-01-01")     # match the single-signal backtest window
HORIZONS = [21, 63]                    # ~1 and ~3 trading months
PRIMARY = 21                           # horizon for the comparison and the tradeable curve

# a backtest client with enough history to span the window (mock) or the live token
bt_client = (QuiverClient(token=QUIVER_TOKEN, mock_history_days=600) if USE_LIVE
             else QuiverClient(mock=True, mock_history_days=1400))
prefetch = {
    "congress":      bt_client.congress_trades(historical=True),
    "gov_contracts": bt_client.gov_contracts(),
    "lobbying":      bt_client.lobbying(),
}
event_signals = {k: signals_all[k] for k in ["congress", "gov_contracts", "lobbying"]}
composite_bt = CompositeSignal(event_signals)
score_fn = composite_bt.score_fn(prefetch=prefetch)
universe = sorted(composite_bt.compute(prefetch=prefetch).index.tolist())

if USE_LIVE:
    from quant_research.backtest.prices import price_history
    prices = price_history(universe[:150], START, pd.Timestamp.today().normalize())
else:
    KW = {"congress": "trades", "gov_contracts": "contracts", "lobbying": "filings"}
    ranks = {nm: s.compute(pd.Timestamp.today(), **{KW[nm]: prefetch[nm]})["score"].rank(pct=True)
             for nm, s in event_signals.items()}
    days = pd.bdate_range(START, pd.Timestamp.today())
    rng = np.random.default_rng(11)
    px = {}
    for t in universe:
        drift = sum(0.0012 * (ranks[nm].get(t, 0.5) - 0.5) for nm in ranks)
        px[t] = 100 * np.exp(np.cumsum(rng.normal(drift, 0.010, len(days))))
    prices = pd.DataFrame(px, index=days)

rebal = pd.bdate_range(prices.index[5], prices.index[-1], freq="W-FRI")
print(f"{len(universe)} names | {prices.shape[1]} priced | {len(rebal)} weekly rebalance dates "
      f"| {prices.index[0].date()} to {prices.index[-1].date()}")

https://api.quiverquant.com/beta/bulk/congresstrading
1360 names | 135 priced | 232 weekly rebalance dates | 2022-01-03 to 2026-06-25


### Statistical view — does the score rank forward returns?

The event study scores the universe on every weekly date and measures forward returns, so
it carries the most observations. Running it at both horizons is the robustness check: an
edge worth trusting should not vanish when the hold lengthens from one month to three. The
overlapping windows mean the *effective* independent sample is well below the raw period
count, so a modest, stable, positive IC is the realistic target.


In [7]:
studies, rows = {}, []
for H in HORIZONS:
    res = event_study(score_fn, prices, rebal, horizon=H, n_buckets=5)
    studies[H] = res
    s = res["summary"]
    rows.append({"horizon_d": H, "periods": s["n_periods"],
                 "mean_ic": round(s["mean_ic"], 3),
                 "ic_pos_%": round(s["ic_positive_share"] * 100, 0),
                 "ls_%": round(s["mean_long_short"] * 100, 3),
                 "ls_pos_%": round(s["ls_positive_share"] * 100, 0)})
print("Composite event study across horizons (overlapping periods):")
pd.DataFrame(rows).set_index("horizon_d")

Composite event study across horizons (overlapping periods):


,periods,mean_ic,ic_pos_%,ls_%,ls_pos_%
horizon_d,,,,,
21,228,0.011,53.0,0.121,51.0
63,219,0.016,52.0,0.459,56.0


#### Does blending beat the components?

The same engine, prices, and dates applied to each single signal at the primary horizon. A
composite IC above every single signal means diversification is paying off. A single signal
leading the composite means that signal carries the edge and the weights deserve a rethink.


In [8]:
KW = {"congress": "trades", "gov_contracts": "contracts", "lobbying": "filings"}

def single_fn(sig, key):
    return lambda d: (lambda r: r["score"] if len(r) else None)(
        sig.compute(d, **{KW[key]: prefetch[key]}))

comp_s = studies[PRIMARY]["summary"]
rows = []
for nm, sig in event_signals.items():
    rs = event_study(single_fn(sig, nm), prices, rebal, horizon=PRIMARY)["summary"]
    rows.append({"signal": nm, "mean_ic": rs["mean_ic"],
                 "ic_pos_%": rs["ic_positive_share"] * 100,
                 "ls_%/mo": rs["mean_long_short"] * 100})
rows.append({"signal": "COMPOSITE", "mean_ic": comp_s["mean_ic"],
             "ic_pos_%": comp_s["ic_positive_share"] * 100,
             "ls_%/mo": comp_s["mean_long_short"] * 100})
print(f"Single signals vs composite at the {PRIMARY}-day horizon:")
pd.DataFrame(rows).set_index("signal").round(3)

Single signals vs composite at the 21-day horizon:


,mean_ic,ic_pos_%,ls_%/mo
signal,,,
congress,0.023,55.702,0.050
gov_contracts,-0.213,21.429,-4.372
lobbying,0.006,48.571,-0.340
COMPOSITE,0.011,52.632,0.121


#### Forward return by score bucket

If the score carries information, the bars rise from the lowest bucket to the highest. Edge
concentrated only in the top bucket still works for picking names, even when the middle is
flat.


In [9]:
import plotly.express as px

bm = studies[PRIMARY]["bucket_means"]
df = bm.reset_index(); df.columns = ["bucket", "fwd"]
df["bucket"] = df["bucket"].map({0: "1 (low)", 1: "2", 2: "3", 3: "4", 4: "5 (high)"})
fig = px.bar(df, x="bucket", y="fwd",
             title=f"Composite — mean {PRIMARY}-day forward return by score bucket",
             labels={"fwd": "mean forward return"})
fig.update_traces(marker_color="#2563eb")
fig.update_layout(height=360, yaxis_tickformat=".1%")
fig.show()

### Tradeable view — what would trading it have earned?

This lens simulates an actual strategy, so entries are spaced a full horizon apart and the
holds do not overlap. The long-short curve longs the top quintile and shorts the bottom,
stripping out market beta to isolate what the signal itself earns; the equal-weight
benchmark is shown alongside for context. Expect far fewer trades here than periods above —
that gap is the difference between measuring a signal and trading it.


In [10]:
ls_equity, ls_rets = long_short_curve(score_fn, prices, rebal, horizon=PRIMARY)
bm_equity, bm_rets = benchmark_curve(prices, rebal, horizon=PRIMARY)

fig = go.Figure()
fig.add_scatter(y=ls_equity.values, name="composite long-short (market-neutral)")
fig.add_scatter(y=bm_equity.values, name="equal-weight benchmark")
fig.update_layout(height=420, title="Composite strategy vs benchmark",
                  xaxis_title="non-overlapping holding period", yaxis_title="growth of $1")
fig.show()

m = metrics(ls_rets, horizon=PRIMARY)
print(f"Long-short composite ({PRIMARY}d holds): "
      f"CAGR {m['cagr']*100:+.1f}% | Sharpe {m['sharpe']:.2f} | "
      f"maxDD {m['max_drawdown']*100:.1f}% | hit {m['hit_rate']:.0%} | "
      f"{m['n_trades']} non-overlapping trades")

Long-short composite (21d holds): CAGR +8.5% | Sharpe 0.48 | maxDD -22.4% | hit 50% | 46 non-overlapping trades


## Reading it honestly

- **The two views answer different questions.** The statistical view, with its hundreds of
  overlapping periods, measures whether the score ranks returns; the tradeable view, with
  its few dozen non-overlapping holds, measures what a real strategy would have earned. The
  large period count and the small trade count are both correct.
- **The horizon table is the robustness check.** An edge that holds at both $21$ and $63$
  days is more credible than one that appears at a single horizon. Watch whether the IC
  stays positive as the hold lengthens.
- **Overlap inflates naive significance.** Weekly rebalances on a multi-week horizon
  overlap, so the effective independent sample sits well below the raw period count. Treat
  a modest, stable, positive IC as success.
- **The composite earns robustness, not a guaranteed IC bump.** When one signal dominates
  the edge, the blend tracks it; the value is that the strategy does not lean on a single
  data source. When signals diversify, the blend can exceed each part.
- **Off-exchange sits out by necessity.** With one day of data it cannot be scored against
  forward returns, so it contributes only to the live ranking above.

**Flagged optimization for later.** Two weighting levers are worth revisiting against this
evidence: the off-exchange DPI-vs-volume balance (currently $0.65/0.35$, footprint-aware),
and the composite weights themselves. If a single signal leads the composite, shifting
weight toward it — or reworking the weaker signals — is the natural next experiment, now
that this notebook makes the comparison measurable across horizons.


## Reweighting the composite

The comparison surfaced a problem the project weights did not anticipate: gov_contracts
ran *anti*-predictive over this window and lobbying added little, so the equal-ish blend
landed below congress alone. This section treats that as a hypothesis to test rather than a
verdict. It tries several weightings of the three event signals and reads the composite's
information coefficient under each. Off-exchange stays out of the backtest (no history), so
these weights cover congress, contracts, and lobbying only.


In [11]:
variants = {
    "baseline (40/30/20)": {"congress": 0.40, "gov_contracts": 0.30, "lobbying": 0.20},
    "congress-leaning":    {"congress": 0.60, "gov_contracts": 0.20, "lobbying": 0.20},
    "drop gov_contracts":  {"congress": 0.70, "gov_contracts": 0.00, "lobbying": 0.30},
    "congress-only":       {"congress": 1.00, "gov_contracts": 0.00, "lobbying": 0.00},
    "fade gov_contracts":  {"congress": 0.50, "gov_contracts": -0.30, "lobbying": 0.20},
}

rows = []
for label, w in variants.items():
    fn = CompositeSignal(event_signals, weights=w).score_fn(prefetch=prefetch)
    s = event_study(fn, prices, rebal, horizon=PRIMARY)["summary"]
    rows.append({"weighting": label,
                 "mean_ic": round(s["mean_ic"], 3),
                 "ic_pos_%": round(s["ic_positive_share"] * 100, 0),
                 "ls_%/mo": round(s["mean_long_short"] * 100, 3)})
print(f"Composite under alternative weightings, {PRIMARY}-day horizon:")
pd.DataFrame(rows).set_index("weighting")

Composite under alternative weightings, 21-day horizon:


,mean_ic,ic_pos_%,ls_%/mo
weighting,,,
baseline (40/30/20),0.011,53.0,0.121
congress-leaning,0.012,54.0,0.027
drop gov_contracts,0.012,54.0,-0.043
congress-only,0.009,53.0,-0.124
fade gov_contracts,0.011,53.0,-0.164


### Choosing a weighting without overfitting

The disciplined choice leans on the signal the evidence supports while resisting the urge
to chase the single highest number. A congress-leaning or gov-dropped blend that recovers
the composite IC toward congress's standalone level is defensible and robust.

The `fade gov_contracts` row — a negative weight that bets the signal works in reverse —
will often post the strongest IC here, and that is precisely the trap. Optimizing a sign
flip on one window is overfitting. Adopting it would require the relationship to hold
across sub-periods and both horizons, plus a credible story for why contract awards should
be faded rather than followed. The honest output of this section is a weighting that leans
on what works, with the fade kept as a hypothesis to test rather than a result to deploy.

The next step that would firm this up is `feature_buckets` on gov_contracts, to see whether
`award_value` or `acceleration` drives the negative relationship, and a sub-period split to
check whether it is regime-bound. Those belong in a follow-up; the takeaway here is that the
backtest changed the weights from an assumption into an evidence-based choice.


### A construction caveat the table exposed

The `congress-only` row repays a second look. It scores around $0.009$, yet congress
measured on its own universe scored higher in the comparison above. They are the same
underlying signal, so the gap is structural: the composite ranks the *union* of every
signal's tickers and fills a name absent from a given signal with $0$. Because the four
datasets cover largely disjoint universes, a congress-weighted composite still carries a
large block of non-congress names pinned at $0$, and that block adds noise to the
cross-sectional ranking, pulling the measured IC down.

The "absent contributes $0$" rule was a deliberate choice to reward corroboration, and it
does that — at the cost of diluting a *selective* signal whose edge lives in a narrow set of
names. The practical reading is that the composite works as a breadth-and-corroboration
lens rather than a way to sharpen a single strong signal. A rank-based composite that
averages a name's percentile only across the signals it actually appears in would sidestep
the dilution, while giving up the corroboration reward — a design trade rather than a free
gain, logged here as a candidate refinement.


## Reconciling with published Congress-tracking returns

Copytrading products and vendor dashboards advertise congressional-trading strategies with
headline CAGRs well above what this backtest shows. The gap comes mostly from a difference
in what is being measured, with the pipeline itself sound.

- **Long-only total return against market-neutral alpha.** The published strategies are
  long-only — they buy what members buy and hold — so their return is dominated by market
  beta, since congressional buyers concentrate in megacap technology and quality growth that
  led the market over the windows those records cover. This notebook reports the
  market-neutral long-short, which strips that beta to isolate alpha. The long-only
  top-quintile view in the optimizations notebook makes the comparison direct and separates
  beta from alpha.
- **Disclosure lag.** Congressional trades become public up to $45$ days after execution,
  and the signal keys on the disclosure date, so it acts on information already weeks old
  with no look-ahead. Much of the short-horizon timing edge has decayed by then, which is why
  the $63$-day horizon read marginally firmer than the $21$-day. A backtest keyed on the
  transaction date captures the move from day zero and looks far stronger, while relying on
  information no participant actually held.
- **Member and window selection.** A single-member tracker reflects someone chosen after the
  fact for having done well, while aggregating every member also carries those who did not.
  The committee-alignment result measured earlier (aligned trades +2.18% against +1.21% for
  the rest) is the survivorship-free version of the idea that some members' trades carry more
  information. Published figures also tend to rest on favorable windows and quote gross
  returns rather than risk-adjusted ones.
- **Universe truncation.** The backtest prices only the first $150$ tickers of the union
  universe, taken alphabetically, so the IC is estimated on an arbitrary slice rather than
  the full ranking. Expanding the priced universe is among the optimizations that follow.

The honest reading: a long-only congressional portfolio genuinely compounded at a high rate,
most of it beta, while the cross-sectional timing alpha measured without look-ahead is thin.
Both hold at once, and surfacing that distinction is the value — it separates a marketing
figure from a risk-adjusted one.


## Commit

In [12]:
!git config --global user.email "aballa1234@gmail.com"
!git config --global user.name "Alex Balla"

!git add -A
!git commit -m "Add composite scorer and backtest"
!git push

[main c003999] Add composite scorer and backtest
 1 file changed, 82 insertions(+)
 create mode 100644 src/quant_research/signals/composite.py
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (6/6), 1.85 KiB | 1.85 MiB/s, done.
Total 6 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/balla-a12/quant-equity-research.git
   4377d99..c003999  main -> main


## Recap and next

The four signals now blend into one conviction ranking, and the backtest puts a number on
whether that blend earns its keep — across two horizons and through both a statistical and a
tradeable lens. The final piece is the **dashboard** (`07`): today's top composite
candidates with the per-signal breakdown, the trending names in each signal, and this
backtest evidence attached, so the whole system reads at a glance.
